In [ ]:
!pip install lmdb

In [ ]:
import os
import lmdb
import cv2
import pandas as pd
import pickle
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_DIR = "/content/drive/MyDrive/deepfake_project"

train_csv = os.path.join(BASE_DIR, "data/metadata/train.csv")
test_csv  = os.path.join(BASE_DIR, "data/metadata/test.csv")

train_df = pd.read_csv(train_csv)
test_df  = pd.read_csv(test_csv)

In [ ]:
def create_lmdb(df, lmdb_path):

    # 32k images → safe buffer (LMDB requires over-allocation)
    map_size = 3 * 1024**3  # 3GB safe margin

    env = lmdb.open(
        lmdb_path,
        map_size=map_size,
        subdir=False,
        meminit=False,
        map_async=True,
        readahead=False,
        lock=True
    )

    txn = env.begin(write=True)

    for i, row in tqdm(df.iterrows(), total=len(df)):

        img_path = row["frame_path"]  # already FULL path in your CSV

        img = cv2.imread(img_path)
        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # fast JPEG encoding (good balance of speed/size)
        success, img_encoded = cv2.imencode(
            ".jpg",
            img,
            [int(cv2.IMWRITE_JPEG_QUALITY), 90]
        )

        if not success:
            continue

        # -----------------------------
        # VIDEO-AWARE STRUCTURE
        # -----------------------------
        video_id = row["video_id"]   # e.g. real/000
        label = float(row["label"])
        frame_name = os.path.basename(img_path)

        # unique LMDB key per frame
        key = f"{video_id}/{frame_name}".encode()

        value = {
            "image": img_encoded.tobytes(),
            "label": label,
            "video_id": video_id,
            "frame_name": frame_name
        }

        txn.put(
            key=key,
            value=pickle.dumps(value, protocol=pickle.HIGHEST_PROTOCOL)
        )

        # periodic commit (prevents RAM buildup)
        if i % 2000 == 0 and i > 0:
            txn.commit()
            txn = env.begin(write=True)

    txn.commit()
    env.close()

    print(f"✔ LMDB created at: {lmdb_path}")

In [ ]:
train_lmdb_path = "/content/train.lmdb"
test_lmdb_path  = "/content/test.lmdb"

In [ ]:
create_lmdb(train_df, train_lmdb_path)
create_lmdb(test_df, test_lmdb_path)

100%|██████████| 25696/25696 [4:48:48<00:00,  1.48it/s]


✔ LMDB created at: /content/train.lmdb


100%|██████████| 6362/6362 [1:10:23<00:00,  1.51it/s]

✔ LMDB created at: /content/test.lmdb


In [ ]:
!cp /content/train.lmdb /content/drive/MyDrive/deepfake_project/
!cp /content/test.lmdb /content/drive/MyDrive/deepfake_project/

IGNORE BELOW, THEY ARE IMPLEMENTED IN **2d_cnn_late_fusion_lmdb** NOTEBOOK

In [ ]:
!pip install timm albumentations lmdb

In [ ]:
import os
import cv2
import lmdb
import timm
import torch
import pickle
import random
import numpy as np
import pandas as pd

from tqdm import tqdm
from collections import defaultdict

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.utils.class_weight import compute_class_weight

import albumentations as A
from albumentations.pytorch import ToTensorV2

from torch import nn
from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

In [ ]:
class CFG:

    IMG_SIZE = 224

    BATCH_SIZE = 32

    EPOCHS = 5

    LR = 1e-4

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    NUM_WORKERS = 2

    TRAIN_LMDB = "/content/train.lmdb"

    TEST_LMDB = "/content/test.lmdb"

In [ ]:
train_transform = A.Compose([

    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),

    A.HorizontalFlip(p=0.5),

    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.5
    ),

    A.ImageCompression(
        quality_range=(60, 100),
        p=0.5
    ),

    A.GaussianBlur(
        blur_limit=(3, 5),
        p=0.3
    ),

    A.Normalize(),

    ToTensorV2()
])

valid_transform = A.Compose([

    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),

    A.Normalize(),

    ToTensorV2()
])

In [ ]:
class DeepfakeLMDBDataset(Dataset):

    def __init__(self, lmdb_path, transform=None):

        self.env = lmdb.open(
            lmdb_path,
            readonly=True,
            lock=False,
            readahead=False,
            meminit=False,
            subdir=False
        )

        self.txn = self.env.begin()

        self.keys = []

        self.labels = []

        with self.env.begin() as txn:

            cursor = txn.cursor()

            for key, value in cursor:

                self.keys.append(key)

                sample = pickle.loads(value)

                self.labels.append(int(sample["label"]))

        self.transform = transform

    def __len__(self):

        return len(self.keys)

    def __getitem__(self, idx):

        key = self.keys[idx]

        value = self.txn.get(key)

        sample = pickle.loads(value)

        img_bytes = sample["image"]

        label = sample["label"]

        video_id = sample["video_id"]

        img_array = np.frombuffer(img_bytes, np.uint8)

        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(image=img)["image"]

        return {
            "image": img,
            "label": torch.tensor(label, dtype=torch.float32),
            "video_id": video_id
        }

In [ ]:
train_dataset = DeepfakeLMDBDataset(
    CFG.TRAIN_LMDB,
    transform=train_transform
)

test_dataset = DeepfakeLMDBDataset(
    CFG.TEST_LMDB,
    transform=valid_transform
)

Error: The environment '/content/train.lmdb' is already open in this process.